# Titanic Data Cleaning and Preprocessing

This notebook completes Edutech Solution Data Analytics Internship Task 2. It handles missing values, removes duplicates, normalizes features, handles outliers, and saves a cleaned Titanic dataset.

## 1. Import Libraries

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

## 2. Load Dataset

In [2]:
BASE_DIR = Path.cwd()
RAW_DATA = BASE_DIR / "data" / "raw" / "Titanic-Dataset.csv"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
CLEAN_DATA = PROCESSED_DIR / "titanic_cleaned.csv"
SUMMARY_FILE = PROCESSED_DIR / "cleaning_summary.txt"

df = pd.read_csv(RAW_DATA)
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 3. Inspect Dataset

In [3]:
print("Shape:", df.shape)
print("Duplicate rows:", df.duplicated().sum())
print("\nMissing values:")
print(df.isna().sum())


Shape: (891, 12)
Duplicate rows: 0

Missing values:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64


## 4. Helper Functions

In [4]:
def cap_iqr_outliers(series):
    """Cap numeric outliers using the 1.5 IQR rule."""
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return series.clip(lower=lower, upper=upper)


def min_max_scale(series):
    min_value = series.min()
    max_value = series.max()
    if np.isclose(max_value, min_value):
        return pd.Series(0, index=series.index)
    return (series - min_value) / (max_value - min_value)

## 5. Clean and Preprocess Data

In [5]:
cleaned_df = df.copy()
original_shape = cleaned_df.shape
original_missing = cleaned_df.isna().sum()

steps = [f"Loaded raw dataset with {original_shape[0]} rows and {original_shape[1]} columns."]

# Remove duplicates
duplicate_count = int(cleaned_df.duplicated().sum())
cleaned_df = cleaned_df.drop_duplicates().reset_index(drop=True)
steps.append(f"Removed {duplicate_count} duplicate rows.")

# Handle missing values
cleaned_df["Age"] = cleaned_df["Age"].fillna(cleaned_df["Age"].median())
cleaned_df["Embarked"] = cleaned_df["Embarked"].fillna(cleaned_df["Embarked"].mode()[0])
cleaned_df["Cabin_Known"] = cleaned_df["Cabin"].notna().astype(int)
cleaned_df["Cabin_Deck"] = cleaned_df["Cabin"].fillna("Unknown").astype(str).str[0]
cleaned_df.loc[cleaned_df["Cabin"].isna(), "Cabin_Deck"] = "Unknown"
steps.append("Handled missing values: Age=median, Embarked=mode, Cabin converted to useful indicators.")

# Feature engineering
cleaned_df["FamilySize"] = cleaned_df["SibSp"] + cleaned_df["Parch"] + 1
cleaned_df["IsAlone"] = (cleaned_df["FamilySize"] == 1).astype(int)
steps.append("Created FamilySize and IsAlone features.")

# Handle outliers
for column in ["Age", "Fare", "SibSp", "Parch", "FamilySize"]:
    cleaned_df[column] = cap_iqr_outliers(cleaned_df[column])
steps.append("Capped numeric outliers with the 1.5 IQR rule.")

# Normalize numeric features
for column in ["Age", "Fare", "SibSp", "Parch", "FamilySize"]:
    cleaned_df[f"{column}_Norm"] = min_max_scale(cleaned_df[column])
steps.append("Normalized numeric features with min-max scaling.")

# Encode categorical columns
cleaned_df["Sex"] = cleaned_df["Sex"].str.lower().map({"male": 0, "female": 1})
embarked_dummies = pd.get_dummies(cleaned_df["Embarked"], prefix="Embarked").astype(int)
deck_dummies = pd.get_dummies(cleaned_df["Cabin_Deck"], prefix="Deck").astype(int)
cleaned_df = pd.concat([cleaned_df, embarked_dummies, deck_dummies], axis=1)
steps.append("Encoded categorical columns: Sex, Embarked, and Cabin deck.")

# Drop raw text columns
drop_columns = ["Name", "Ticket", "Cabin", "Embarked", "Cabin_Deck"]
cleaned_df = cleaned_df.drop(columns=drop_columns)
steps.append(f"Dropped high-cardinality/raw text columns: {', '.join(drop_columns)}.")

missing_after = int(cleaned_df.isna().sum().sum())
steps.append(f"Final cleaned dataset has {cleaned_df.shape[0]} rows and {cleaned_df.shape[1]} columns.")
steps.append(f"Missing values before cleaning: {int(original_missing.sum())}; after cleaning: {missing_after}.")

cleaned_df.head()

,PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Cabin_Known,FamilySize,...,Embarked_S,Deck_A,Deck_B,Deck_C,Deck_D,Deck_E,Deck_F,Deck_G,Deck_T,Deck_Unknown
0,1,0,3,0,22.0,1.0,0,7.2500,0,2.0,...,1,0,0,0,0,0,0,0,0,1
1,2,1,1,1,38.0,1.0,0,65.6344,1,2.0,...,0,0,0,1,0,0,0,0,0,0
2,3,1,3,1,26.0,0.0,0,7.9250,0,1.0,...,1,0,0,0,0,0,0,0,0,1
3,4,1,1,1,35.0,1.0,0,53.1000,1,2.0,...,1,0,0,1,0,0,0,0,0,0
4,5,0,3,0,35.0,0.0,0,8.0500,0,1.0,...,1,0,0,0,0,0,0,0,0,1


## 6. Save Cleaned Dataset and Summary

In [6]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
cleaned_df.to_csv(CLEAN_DATA, index=False)

summary_lines = ["Titanic Data Cleaning Summary", "=" * 30, ""]
summary_lines.extend(f"- {step}" for step in steps)
summary_lines.extend(["", "Final columns:", ", ".join(cleaned_df.columns)])
SUMMARY_FILE.write_text("\n".join(summary_lines), encoding="utf-8")

print(f"Saved cleaned dataset to: {CLEAN_DATA}")
print(f"Saved cleaning summary to: {SUMMARY_FILE}")

Saved cleaned dataset to: c:\Users\HP\Desktop\data science\Titanic-dataset-preprocessing\data\processed\titanic_cleaned.csv
Saved cleaning summary to: c:\Users\HP\Desktop\data science\Titanic-dataset-preprocessing\data\processed\cleaning_summary.txt


## 7. Verify Final Dataset

In [7]:
print("Final shape:", cleaned_df.shape)
print("Missing values:", int(cleaned_df.isna().sum().sum()))
print("Object/text columns:", list(cleaned_df.select_dtypes(include="object").columns))
cleaned_df.head()

Final shape: (891, 28)
Missing values: 0
Object/text columns: []


,PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Cabin_Known,FamilySize,...,Embarked_S,Deck_A,Deck_B,Deck_C,Deck_D,Deck_E,Deck_F,Deck_G,Deck_T,Deck_Unknown
0,1,0,3,0,22.0,1.0,0,7.2500,0,2.0,...,1,0,0,0,0,0,0,0,0,1
1,2,1,1,1,38.0,1.0,0,65.6344,1,2.0,...,0,0,0,1,0,0,0,0,0,0
2,3,1,3,1,26.0,0.0,0,7.9250,0,1.0,...,1,0,0,0,0,0,0,0,0,1
3,4,1,1,1,35.0,1.0,0,53.1000,1,2.0,...,1,0,0,1,0,0,0,0,0,0
4,5,0,3,0,35.0,0.0,0,8.0500,0,1.0,...,1,0,0,0,0,0,0,0,0,1
